# Computational burden: 27-bus vs 118-bus on one GPU

Standalone notebook. **Runtime -> Change runtime type -> GPU, then Run all.**
Nothing else needs to have been run first.

This answers reviewer R3 #5, which asked for exact training time, hardware and memory. It
trains one 150-epoch seed of each system on the SAME device so the two are directly
comparable, and records wall-clock time plus peak GPU memory. About 8-10 minutes.

Everything else R3 #5 asks for -- dataset-generation cost, per-contingency inference latency,
resident memory, parameter count -- is already measured on CPU and needs no GPU.

The 27-bus run doubles as an independent reproduction of the paper's headline 98.1%.

In [ ]:
# --- Google Drive: all inputs and outputs live in the Drive folder, nothing is uploaded by hand ---
from google.colab import drive
import glob, os, sys, json, time
drive.mount('/content/drive')

cand = glob.glob('/content/drive/MyDrive/**/smart_load_shield_boost', recursive=True)
FOLDER = cand[0] if cand else '/content/drive/MyDrive/smart_load_shield_boost'
OUT_DIR = os.path.join(FOLDER, 'scale118')
os.makedirs(OUT_DIR, exist_ok=True)
sys.path.insert(0, FOLDER)

# v2 = the corrected corpus (schedule ceiling 1.04, operating-point admission test,
# two-sided Eq. (risk) enforced in code).  The v1 files are EVIDENCE for the response
# letter and must survive: nothing here writes to a v1 name.
NPZ   = os.path.join(OUT_DIR, 'contingency_data_118_v2.npz')
SPLIT = os.path.join(OUT_DIR, 'grouped_split_118_v2.npz')

V1 = os.path.join(OUT_DIR, 'contingency_data_118.npz')
assert NPZ != V1 and SPLIT != V1
if os.path.exists(V1):
    print('v1 corpus present and will NOT be touched:', V1)

print('Drive folder :', FOLDER)
print('118-bus dir  :', OUT_DIR)
print('existing files:', sorted(os.listdir(OUT_DIR)) or '(empty)')

In [ ]:
# --- dependencies: Colab has torch; pandapower is pinned to the tested version ---
PANDAPOWER_VERSION = '3.4.0'      # the version this pipeline was verified against
try:
    import pandapower
    assert pandapower.__version__ == PANDAPOWER_VERSION
    print('pandapower', pandapower.__version__, '(already present)')
except (ImportError, AssertionError):
    !pip -q install pandapower=={PANDAPOWER_VERSION}
    import importlib, pandapower
    importlib.reload(pandapower)
    print('pandapower', pandapower.__version__, '(installed)')

import torch, numpy as np
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

In [ ]:
# --- inputs: 27-bus assets sit in the Drive folder root, 118-bus in scale118/ ---
NPZ27   = os.path.join(FOLDER, 'contingency_data.npz')
SPLIT27 = os.path.join(FOLDER, 'grouped_split.npz')
for f in (NPZ27, SPLIT27, NPZ, SPLIT, os.path.join(FOLDER, 'boost_core.py')):
    assert os.path.exists(f), f'missing {f}'
print('all five inputs present')
print('  27-bus :', NPZ27)
print('  118-bus:', NPZ)

## Model and training modules

Inlined from the locally tested `core118.py` and `train118.py`. The 27-bus half uses `boost_core.py` read from the Drive folder.

In [ ]:
"""
Memory-efficient core for scaling the contingency-screening surrogate to the
IEEE 118-bus system.

WHY THIS FILE EXISTS
--------------------
`boost_core.py` materialises, for the WHOLE dataset, a dense edge tensor
[M, N, N, 3] and a dense adjacency [M, N, N].  At N=27 that is 262 MB + 87 MB
and fits on any GPU.  At N=118 the same tensors are 5.0 GB + 1.7 GB, and the
attention layer's [B, N, N, heads, 2d] concatenation adds ~900 MB per layer at
batch 64.  A naive port therefore OOMs before the first step.

Three changes fix it, and all three are EXACTLY equivalent to the original --
this is the whole point, and `test_equivalence.py` proves it numerically:

  1. SPARSE EDGE ENCODING.  The original computes `edge_enc` on every one of the
     N^2 dense cells, multiplies by a base mask that zeroes non-branch cells,
     then sums over j.  Only branch cells survive, so we instead evaluate
     `edge_enc` on the ~186 real branch cells and `index_add` the result onto
     both endpoints.  Identical output, O(n_branch) instead of O(N^2) memory.

  2. FACTORISED ATTENTION.  The original builds [B,N,N,heads,2d] to compute
     `attn(cat([h_i, h_j]))`.  Because `attn` is linear and bias-free,
        attn([h_i; h_j]) = a_src . h_i + a_dst . h_j,
     with `a_src`/`a_dst` the two halves of the SAME weight matrix.  We compute
     two [B,N,heads] vectors and broadcast-add, then aggregate with an einsum
     that never materialises the product.  Identical output, O(N^2 * heads)
     instead of O(N^2 * 2d) memory -- a 16x reduction at heads=4, d=32.

  3. PER-BATCH DENSE ADJACENCY.  Built on the fly from the compact in-service
     vector instead of being precomputed for all M samples.

PARALLEL BRANCHES
-----------------
case118 has 7 pairs of buses joined by two branches each; the 26/27-bus case has
none.  The original dense code would silently let one parallel branch overwrite
the other (last write wins), which is both arbitrary and physically wrong.  Here
a graph EDGE is a unique bus pair ("cell") and parallel branches are aggregated
onto it: features are averaged, and the pair is adjacent if ANY of its branches
is in service.  On a network without parallel branches every cell holds exactly
one branch, so this reduces EXACTLY to the original behaviour -- which is why
the 27-bus equivalence test is a valid gate for the 118-bus code path.

Contingencies are still defined per BRANCH (186 of them), so tripping one of two
parallel circuits is a genuinely milder, and correctly represented, event.
"""
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================================
# Graph context: branch list -> unique undirected cells
# ============================================================================
def build_graph_ctx(edge_pairs, n_bus):
    """
    edge_pairs : [n_branch, 2] int array of (from_node, to_node) INDEX pairs
                 (already 0..n_bus-1, i.e. positions in the node ordering).

    Returns a dict with
      cell_of_branch : [n_branch] long, which cell each branch belongs to
      cell_i, cell_j : [n_cell] long, the two endpoints of each cell
      branch_per_cell: [n_cell] float, how many branches share the cell
      n_cell         : int
    Cells are emitted in order of first appearance so that, on a network with no
    parallel branches, cell k IS branch k and the mapping is the identity.
    """
    ep = np.asarray(edge_pairs).astype(np.int64)
    order, cell_of_branch = {}, np.zeros(len(ep), np.int64)
    ci, cj = [], []
    for k, (f, t) in enumerate(ep):
        key = (int(min(f, t)), int(max(f, t)))
        if key not in order:
            order[key] = len(ci)
            ci.append(key[0])
            cj.append(key[1])
        cell_of_branch[k] = order[key]
    n_cell = len(ci)
    cnt = np.zeros(n_cell, np.float32)
    for c in cell_of_branch:
        cnt[c] += 1.0
    return dict(
        cell_of_branch=torch.tensor(cell_of_branch, dtype=torch.long),
        cell_i=torch.tensor(ci, dtype=torch.long),
        cell_j=torch.tensor(cj, dtype=torch.long),
        branch_per_cell=torch.tensor(cnt, dtype=torch.float32),
        n_cell=n_cell,
        n_bus=int(n_bus),
        n_branch=len(ep),
    )


def cells_and_adj(ef, ins, ctx):
    """
    Compact per-branch tensors -> per-cell features and dense adjacency.

    ef  : [B, n_branch, 2]  base-case (loading, P_from) per branch
    ins : [B, n_branch]     in-service flag per branch (the contingency)

    Returns
      cell_feat : [B, n_cell, 3]  averaged (loading, P_from, in-service) per cell
      adj       : [B, N, N]       1 where a cell has ANY branch in service, plus
                                  self-loops -- matching the original construction
    """
    B = ef.shape[0]
    dev = ef.device
    cob = ctx['cell_of_branch'].to(dev)
    ci, cj = ctx['cell_i'].to(dev), ctx['cell_j'].to(dev)
    cnt = ctx['branch_per_cell'].to(dev)
    N, n_cell = ctx['n_bus'], ctx['n_cell']

    feat = torch.cat([ef, ins.unsqueeze(-1)], -1)                  # [B,n_branch,3]
    csum = torch.zeros(B, n_cell, 3, device=dev, dtype=feat.dtype)
    csum.index_add_(1, cob, feat)
    cell_feat = csum / cnt.view(1, -1, 1)                          # mean over parallels

    isum = torch.zeros(B, n_cell, device=dev, dtype=ins.dtype)
    isum.index_add_(1, cob, ins)
    live = (isum > 0).to(ef.dtype)                                 # OR over parallels

    adj = torch.zeros(B, N, N, device=dev, dtype=ef.dtype)
    adj[:, ci, cj] = live
    adj[:, cj, ci] = live
    adj[:, torch.arange(N, device=dev), torch.arange(N, device=dev)] = 1.0
    return cell_feat, adj


def dense_from_compact(ef, ins, edge_pairs, n_bus):
    """
    Reference implementation: reproduce boost_core's dense tensors exactly.
    Used only by the equivalence test and by the 27-bus reference path.
    """
    B = ef.shape[0]
    dev = ef.device
    pairs = torch.as_tensor(np.asarray(edge_pairs).astype(np.int64), device=dev)
    fr, to = pairs[:, 0], pairs[:, 1]
    N = n_bus
    dense = torch.zeros(B, N, N, 3, device=dev, dtype=ef.dtype)
    adj = torch.zeros(B, N, N, device=dev, dtype=ef.dtype)
    dense[:, fr, to, :2] = ef
    dense[:, to, fr, :2] = ef
    dense[:, fr, to, 2] = ins
    dense[:, to, fr, 2] = ins
    adj[:, fr, to] = ins
    adj[:, to, fr] = ins
    adj[:, torch.arange(N, device=dev), torch.arange(N, device=dev)] = 1.0
    base = torch.zeros(N, N, device=dev, dtype=ef.dtype)
    base[fr, to] = 1.0
    base[to, fr] = 1.0
    return dense, adj, base


# ============================================================================
# Attention
# ============================================================================
class GATFast(nn.Module):
    """
    Parameter-identical to boost_core.GAT (`W`, `attn`), so state dicts are
    interchangeable, but computed without the [B,N,N,heads,2d] concatenation.
    """

    def __init__(self, in_dim, out_dim, heads=4, dropout=0.15):
        super().__init__()
        self.heads, self.d = heads, out_dim // heads
        self.W = nn.Linear(in_dim, out_dim, bias=False)
        self.attn = nn.Linear(2 * self.d, 1, bias=False)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, adj):                      # x:[B,N,D]  adj:[B,N,N]
        B, N, _ = x.shape
        h = self.W(x).view(B, N, self.heads, self.d)
        w = self.attn.weight.view(-1)                                   # [2d]
        a_src, a_dst = w[: self.d], w[self.d:]
        es = torch.einsum('bnhd,d->bnh', h, a_src)                      # a_src . h_i
        et = torch.einsum('bnhd,d->bnh', h, a_dst)                      # a_dst . h_j
        e = F.leaky_relu(es.unsqueeze(2) + et.unsqueeze(1), 0.2)        # [B,N,N,heads]
        e = e.masked_fill((adj == 0).unsqueeze(-1), float('-inf'))
        a = self.drop(torch.nan_to_num(F.softmax(e, dim=2), 0))
        return torch.einsum('bijh,bjhd->bihd', a, h).reshape(B, N, -1)


class GradScale(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, g):
        return g * ctx.alpha, None


def gs(x, alpha):
    return GradScale.apply(x, alpha) if alpha < 1.0 else x


# ============================================================================
# Model
# ============================================================================
class CSGNN118(nn.Module):
    """
    Parameter-identical to boost_core.CSGNNv2; only the edge encoding and the
    attention arithmetic differ.  `set_graph()` replaces `set_base_mask()`.
    """

    def __init__(self, node_dim=4, edge_dim=3, hidden=128, heads=4, dropout=0.15,
                 n_gat=3, branched=True, uncert=False, stack=False, vbus=False,
                 vmax=False):
        """
        vmax: add a V_max regression head mirroring the V_min head.

        Every other auxiliary head is undervoltage-only (V_min, 1[V_i<0.95],
        divergence), which matched Eq. (risk) while the labelling code was
        one-sided.  With the two-sided rule enforced, a contingency can be
        Unstable through V_max >= 1.05 while its V_min is perfectly healthy, and
        nothing in the network represents that: on the 118-bus corpus 82 of 83
        false-safe cases were exactly this.  This head supplies the missing
        signal.  It is REGRESSION, so it trains on every converged scenario
        rather than on the handful of overvoltage positives.

        Off by default: the 27-bus corpus never exceeds 1.045 p.u., so the head
        would supervise a constant there, and leaving it off keeps the deployed
        configuration and test_equivalence.py byte-identical.
        """
        super().__init__()
        self.branched, self.uncert, self.stack, self.vbus = branched, uncert, stack, vbus
        self.vmax = vmax
        self.ctx = None
        self.node_enc = nn.Sequential(nn.Linear(node_dim, hidden), nn.GELU(), nn.LayerNorm(hidden))
        self.edge_enc = nn.Sequential(nn.Linear(edge_dim, hidden), nn.GELU(), nn.LayerNorm(hidden))
        self.fuse = nn.Sequential(nn.Linear(hidden * 2, hidden), nn.GELU(), nn.LayerNorm(hidden))
        n_sh = (n_gat - 1) if branched else n_gat
        self.gats = nn.ModuleList([GATFast(hidden, hidden, heads, dropout) for _ in range(n_sh)])
        self.norms = nn.ModuleList([nn.LayerNorm(hidden) for _ in range(n_sh)])
        if branched:
            self.gat_r = GATFast(hidden, hidden, heads, dropout)
            self.norm_r = nn.LayerNorm(hidden)
            self.ga_r = nn.Linear(hidden, 1)
        self.ga = nn.Linear(hidden, 1)
        self.ga_a = nn.Linear(hidden, 1)
        rd = hidden * 3
        n_sf = ((9 if vbus else 6) + (2 if vmax else 0)) if stack else 0
        self.risk_h = nn.Sequential(nn.Linear(rd + n_sf, 64), nn.GELU(), nn.Dropout(dropout), nn.Linear(64, 3))
        self.vmin_h = nn.Sequential(nn.Linear(rd, 64), nn.GELU(), nn.Dropout(dropout), nn.Linear(64, 1))
        if vmax:
            self.vmax_h = nn.Sequential(nn.Linear(rd, 64), nn.GELU(), nn.Dropout(dropout), nn.Linear(64, 1))
        self.vuln_h = nn.Sequential(nn.Linear(hidden, 32), nn.GELU(), nn.Linear(32, 1))
        self.div_h = nn.Sequential(nn.Linear(rd, 64), nn.GELU(), nn.Linear(64, 1))
        if vbus:
            self.vbus_h = nn.Sequential(nn.Linear(hidden, 32), nn.GELU(), nn.Linear(32, 1))
        if uncert:
            self.log_s = nn.Parameter(torch.zeros(4))

    def set_graph(self, ctx):
        """Attach the graph context (cell mapping). Not part of the state dict."""
        self.ctx = ctx

    def _pool(self, h, ga):
        a = torch.softmax(ga(h).squeeze(-1), 1).unsqueeze(-1)
        return torch.cat([h.mean(1), h.max(1).values, (h * a).sum(1)], -1)

    def _edge_context(self, cell_feat):
        """edge_enc on real branch cells only, scattered onto both endpoints."""
        B = cell_feat.shape[0]
        dev = cell_feat.device
        ci, cj = self.ctx['cell_i'].to(dev), self.ctx['cell_j'].to(dev)
        em = self.edge_enc(cell_feat)                                   # [B,n_cell,H]
        ec = torch.zeros(B, self.ctx['n_bus'], em.shape[-1], device=dev, dtype=em.dtype)
        ec.index_add_(1, ci, em)
        ec.index_add_(1, cj, em)
        return ec

    def forward(self, nf, cell_feat, adj, alpha=1.0, beta=1.0):
        h = self.fuse(torch.cat([self.node_enc(nf), self._edge_context(cell_feat)], -1))
        for g, n in zip(self.gats, self.norms):
            h = n(h + g(h, adj))
        ha = gs(h, alpha)
        pa = self._pool(ha, self.ga_a)
        vmin = self.vmin_h(pa).squeeze(-1)
        vmx = self.vmax_h(pa).squeeze(-1) if self.vmax else None
        vuln = self.vuln_h(ha).squeeze(-1)
        div = self.div_h(pa).squeeze(-1)
        vb = (self.vbus_h(ha).squeeze(-1) + 1.0) if self.vbus else None
        if self.branched:
            hr = self.norm_r(h + self.gat_r(h, adj))
            pr = self._pool(hr, self.ga_r)
        else:
            pr = self._pool(h, self.ga)
        if self.stack:
            feats = [vmin, (vmin - 0.95) * 20., (vmin - 0.90) * 20., torch.sigmoid(div),
                     torch.sigmoid(vuln).mean(1), torch.sigmoid(vuln).amax(1)]
            if vb is not None:
                vbm = vb.amin(1)
                feats += [vbm, (vbm - 0.95) * 20., (vbm - 0.90) * 20.]
            if vmx is not None:
                feats += [vmx, (vmx - 1.05) * 20.]
            sf = torch.stack(feats, -1).detach() * beta
            risk = self.risk_h(torch.cat([pr, sf], -1))
        else:
            risk = self.risk_h(pr)
        # Variable arity is deliberate: with vmax off the return is byte-identical
        # to the deployed configuration, so every existing caller and
        # test_equivalence.py keep working untouched.
        if self.vmax:
            return risk, vmin, vuln, div, vb, vmx
        return risk, vmin, vuln, div, vb

In [ ]:
"""
Training + evaluation for the IEEE 118-bus contingency-screening surrogate.

Recipe is the 27-bus one (150-epoch warmup-cosine, label smoothing, class
weights, EMA, risk-gated checkpoint selection on zero false-safe); only the
batch assembly differs, because the dense edge/adjacency tensors are built per
batch instead of for the whole dataset (see core118.py).

Metrics deliberately go beyond the 27-bus set, to pre-empt reviewer questions:
  * PR-AUC alongside ROC-AUC for the per-bus vulnerability head.  On 118 buses
    most buses are non-vulnerable under most contingencies, so ROC-AUC alone
    flatters the model.  This is reviewer R4 #3.
  * A screening benchmark that reports the speed-up BOTH with and without
    feature preprocessing, and that SCORES the sweep, which is reviewer R1 #6.
  * An explicit breakdown of the Unstable class into voltage violation,
    islanding, and non-islanded divergence, which is reviewer R1 #4.
"""
import json
import math
import time
from copy import deepcopy

import numpy as np
import torch
import torch.nn.functional as F


# ============================================================================
# Data
# ============================================================================
def build_data(npz_path, split_path, dev='cpu'):
    """Compact tensors only; dense per-batch structures are built in `batch()`."""
    D = np.load(npz_path)
    S = np.load(split_path)
    n_bus = int(D['n_bus'])
    ctx = build_graph_ctx(D['edge_pairs'], n_bus)

    nmean = torch.tensor(S['nmean']).float()
    nstd = torch.tensor(S['nstd']).float()
    emean = torch.tensor(S['emean']).float()
    estd = torch.tensor(S['estd']).float()

    NF = ((torch.tensor(D['node_feats']) - nmean) / nstd).float().to(dev)
    EF = ((torch.tensor(D['edge_feats']) - emean) / estd).float().to(dev)
    INS = torch.tensor(D['in_service']).float().to(dev)

    d = dict(
        NF=NF, EF=EF, INS=INS, ctx=ctx, n_bus=n_bus, dev=dev,
        Tr=torch.tensor(D['t_risk']).to(dev),
        Tv=torch.tensor(D['t_vmin']).float().to(dev),
        Tu=torch.tensor(D['t_vuln']).float().to(dev),
        Td=torch.tensor(D['t_div']).float().to(dev),
        Tb=torch.tensor(D['t_vbus']).float().to(dev) if 't_vbus' in D else None,
        # V_max target for the high-side head.  Diverged scenarios store an
        # all-zero voltage vector, so this is 0.0 there exactly as t_vmin is,
        # and the loss masks them out the same way.
        Tx=(torch.tensor(D['t_vbus']).float().max(dim=1).values.to(dev)
            if 't_vbus' in D else None),
        idx_tr=torch.tensor(S['idx_tr'].astype('int64')).to(dev),
        idx_va=torch.tensor(S['idx_va'].astype('int64')).to(dev),
        idx_te=torch.tensor(S['idx_te'].astype('int64')).to(dev),
        opcond=D['opcond'],
        # Kept so that inference-time feature extraction uses the SAME statistics
        # the model was trained with.  Feeding raw features to a model trained on
        # normalised ones is silent and produces meaningless predictions.
        nmean=nmean.to(dev), nstd=nstd.to(dev),
        emean=emean.to(dev), estd=estd.to(dev),
    )
    cnt = torch.bincount(d['Tr'][d['idx_tr']], minlength=3).float()
    d['cw'] = (cnt.sum() / (3 * cnt.clamp(min=1))).to(dev)
    return d


def batch(d, b):
    """Assemble one batch: node features + per-cell edge features + adjacency."""
    cell_feat, adj = cells_and_adj(d['EF'][b], d['INS'][b], d['ctx'])
    return d['NF'][b], cell_feat, adj


# ============================================================================
# Evaluation
# ============================================================================
@torch.no_grad()
def evaluate(net, d, idx, amp=False, bs=1024, full=False):
    net.eval()
    P, LOG, VMp, VMt, Dp, Dt, VUp, VUt = [], [], [], [], [], [], [], []
    for s0 in range(0, len(idx), bs):
        b = idx[s0:s0 + bs]
        nf, cf, adj = batch(d, b)
        with torch.autocast('cuda', enabled=amp):
            _o = net(nf, cf, adj)
            r, vm, vu, dv = _o[0], _o[1], _o[2], _o[3]
        r = r.float()
        P.append(r.argmax(1))
        LOG.append(r)
        conv = d['Td'][b] == 0
        VMp.append(vm.float()[conv]); VMt.append(d['Tv'][b][conv])
        Dp.append(torch.sigmoid(dv.float())); Dt.append(d['Td'][b])
        VUp.append(torch.sigmoid(vu.float())[conv]); VUt.append(d['Tu'][b][conv])

    P = torch.cat(P); R = d['Tr'][idx]; LOG = torch.cat(LOG)
    acc = (P == R).float().mean().item()
    fs_n = int((((R == 2) & (P == 0))).sum())
    fs = fs_n / max(1, int((R == 2).sum()))
    vmp, vmt = torch.cat(VMp), torch.cat(VMt)
    vmae = (vmp - vmt).abs().mean().item() if len(vmp) else float('nan')
    ss_res = ((vmp - vmt) ** 2).sum().item()
    ss_tot = ((vmt - vmt.mean()) ** 2).sum().item()
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else float('nan')
    dp, dt = torch.cat(Dp), torch.cat(Dt)
    dpred = (dp > 0.5).float()
    tp = float(((dpred == 1) & (dt == 1)).sum()); fp = float(((dpred == 1) & (dt == 0)).sum())
    fn = float(((dpred == 0) & (dt == 1)).sum())
    prec = tp / (tp + fp) if tp + fp else float('nan')
    rec = tp / (tp + fn) if tp + fn else float('nan')
    f1 = 2 * prec * rec / (prec + rec) if prec and rec and prec + rec else float('nan')

    out = dict(acc=acc, false_safe=fs, false_safe_n=fs_n,
               n_unstable=int((R == 2).sum()),
               vmin_mae=vmae, vmin_r2=r2, div_f1=f1,
               div_precision=prec, div_recall=rec, n=len(idx))
    if not full:
        return out

    # ---- per-class recall + confusion
    conf = np.zeros((3, 3), int)
    for t in range(3):
        for p in range(3):
            conf[t, p] = int(((R == t) & (P == p)).sum())
    out['confusion'] = conf.tolist()
    out['per_class_recall'] = [float(conf[t, t] / max(1, conf[t].sum())) for t in range(3)]

    # ---- calibration on the risk head
    prob = torch.softmax(LOG, 1)
    conf_max, pred = prob.max(1)
    correct = (pred == R).float()
    bins = torch.linspace(0, 1, 16, device=prob.device)
    ece = 0.0
    for i in range(15):
        m = (conf_max > bins[i]) & (conf_max <= bins[i + 1])
        if m.any():
            ece += (m.float().mean() * (correct[m].mean() - conf_max[m].mean()).abs()).item()
    out['ece'] = ece
    onehot = F.one_hot(R, 3).float()
    out['brier'] = ((prob - onehot) ** 2).sum(1).mean().item()
    out['nll'] = F.nll_loss(torch.log(prob.clamp(min=1e-12)), R).item()

    # ---- vulnerability head: ROC-AUC *and* PR-AUC (reviewer R4 #3)
    # NOTE this positive rate is over CONVERGED scenarios only, because the
    # vulnerability target is undefined when the power flow diverged.  The
    # dataset-summary figure is over all scenarios, so the two differ.
    vup = torch.cat(VUp).reshape(-1).cpu().numpy()
    vut = torch.cat(VUt).reshape(-1).cpu().numpy()
    out['vuln_positive_rate_converged'] = float(vut.mean())
    try:
        from sklearn.metrics import roc_auc_score, average_precision_score
        if vut.min() != vut.max():
            out['vuln_roc_auc'] = float(roc_auc_score(vut, vup))
            out['vuln_pr_auc'] = float(average_precision_score(vut, vup))
    except Exception:
        out['vuln_roc_auc'] = out['vuln_pr_auc'] = float('nan')
    try:
        from sklearn.metrics import roc_auc_score
        yn = dt.cpu().numpy(); pn_ = dp.cpu().numpy()
        if yn.min() != yn.max():
            out['div_roc_auc'] = float(roc_auc_score(yn, pn_))
    except Exception:
        pass
    return out


# ============================================================================
# Training
# ============================================================================
def train_eval(d, seed=0, alpha=0.0, branched=False, stack=False, vbus=False, vmax=False,
               hidden=128, heads=4, n_gat=3, epochs=150, bs=256, lr=3e-3, ls=0.05,
               amp=True, verbose=True, eval_bs=1024, log_every=10):
    torch.manual_seed(seed); np.random.seed(seed)
    dev = d['dev']
    amp = amp and str(dev).startswith('cuda')
    vbus = vbus and d.get('Tb') is not None
    vmax = vmax and d.get('Tx') is not None

    m = CSGNN118(hidden=hidden, heads=heads, n_gat=n_gat, branched=branched,
                 stack=stack, vbus=vbus, vmax=vmax).to(dev)
    m.set_graph(d['ctx'])
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-4)
    warm = max(1, epochs // 30)
    sch = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda e: min((e + 1) / warm, 1.0)
        * 0.5 * (1 + math.cos(math.pi * max(0, e - warm) / max(1, epochs - warm))))
    ema = torch.optim.swa_utils.AveragedModel(
        m, multi_avg_fn=torch.optim.swa_utils.get_ema_multi_avg_fn(0.999))
    scaler = torch.amp.GradScaler('cuda', enabled=amp)
    idx_tr = d['idx_tr']
    best, best_state = -1.0, None
    stack_warm = max(1, epochs // 15)
    n_bad = 0
    t0 = time.time()

    for ep in range(epochs):
        m.train()
        beta = min(1.0, (ep + 1) / stack_warm) if stack else 1.0
        perm = idx_tr[torch.randperm(len(idx_tr), device=dev)]
        for s0 in range(0, len(perm), bs):
            b = perm[s0:s0 + bs]
            nf, cf, adj = batch(d, b)
            with torch.autocast('cuda', enabled=amp):
                out = m(nf, cf, adj, alpha=alpha, beta=beta)
                r, vm, vu, dv, vb = out[:5]
                vx = out[5] if len(out) > 5 else None
                conv = d['Td'][b] == 0
                l_risk = F.cross_entropy(r, d['Tr'][b], weight=d['cw'], label_smoothing=ls)
                l_vm = F.smooth_l1_loss(vm[conv], d['Tv'][b][conv]) if conv.any() else r.sum() * 0
                l_vu = (F.binary_cross_entropy_with_logits(vu[conv], d['Tu'][b][conv])
                        if conv.any() else r.sum() * 0)
                l_dv = F.binary_cross_entropy_with_logits(dv, d['Td'][b])
                l_vb = (F.smooth_l1_loss(vb[conv], d['Tb'][b][conv])
                        if (vbus and conv.any()) else r.sum() * 0)
                # same weight as the V_min head: the criterion is two-sided,
                # so the supervision is symmetric
                l_vx = (F.smooth_l1_loss(vx[conv], d['Tx'][b][conv])
                        if (vx is not None and conv.any()) else r.sum() * 0)
                loss = (l_risk + 0.5 * l_vm + 0.5 * l_vx
                        + 0.3 * l_vu + 0.3 * l_dv + 0.3 * l_vb)
            if not torch.isfinite(loss):
                n_bad += 1
                opt.zero_grad(set_to_none=True)
                continue
            opt.zero_grad(); scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(m.parameters(), 2.0)
            scaler.step(opt); scaler.update()
            ema.update_parameters(m)
        sch.step()

        if ep >= epochs // 3 and (ep % 5 == 0 or ep == epochs - 1):
            for cand in (m, ema.module):
                r = evaluate(cand, d, d['idx_va'], amp=amp, bs=eval_bs)
                if r['false_safe'] == 0.0 and r['acc'] > best:
                    best = r['acc']
                    best_state = {k: v.clone() for k, v in cand.state_dict().items()}
        if verbose and (ep % log_every == 0 or ep == epochs - 1):
            r = evaluate(m, d, d['idx_va'], amp=amp, bs=eval_bs)
            print(f"  ep{ep:3d}  val acc {r['acc']*100:5.2f}%  fs {r['false_safe']*100:5.2f}%  "
                  f"best@fs0 {max(best,0)*100:5.2f}%  [{time.time()-t0:.0f}s]")

    if best_state is None:
        best_state = max(((evaluate(c, d, d['idx_va'], amp=amp, bs=eval_bs)['acc'], i, c)
                          for i, c in enumerate((m, ema.module))), key=lambda t: t[0])[2].state_dict()
        best_state = {k: v.clone() for k, v in best_state.items()}
    ema.module.load_state_dict(best_state)
    te = evaluate(ema.module, d, d['idx_te'], amp=amp, bs=eval_bs, full=True)
    te['params'] = sum(p.numel() for p in m.parameters())
    te['epochs'] = epochs
    te['batch_size'] = bs
    te['train_seconds'] = time.time() - t0
    te['nonfinite_steps_skipped'] = n_bad
    return te, ema.module


# ============================================================================
# Screening benchmark + sweep scoring
#   reviewer R1 #6: is preprocessing inside the reported speed-up?
# ============================================================================
@torch.no_grad()
def screening_benchmark(net, d, gen_module, n_repeat=3, verbose=True,
                        load_s=1.0, dv=0.0, pv_mw=300.0):
    """
    Time AND score a full N-1 sweep.

    ORACLE    : one AC power flow per contingency.  Also supplies ground truth.
    SURROGATE : (a) forward pass only, (b) end to end, including the base power
                flow, feature extraction and -- critically -- the SAME
                normalisation the model was trained with.

    Reporting both timings is the honest answer to "is preprocessing inside the
    speed-up?".  Scoring the sweep is what backs the claim that the screen flags
    every unstable contingency it screens: `sweep_false_safe` counts the truly
    unstable contingencies the surrogate cleared as Stable.
    """
    import pandapower as pp
    net0, pv_idx = gen_module.build_net()
    branches = ([('line', i) for i in net0.line.index]
                + [('trafo', i) for i in net0.trafo.index])
    n_branch = len(branches)
    bus_ids = sorted(net0.bus.index.tolist())
    base_p = net0.load['p_mw'].copy()
    base_q = net0.load['q_mvar'].copy()
    base_vg = net0.gen['vm_pu'].copy()
    base_vs = net0.ext_grid['vm_pu'].copy()
    pv_tbl = 'sgen' if gen_module.PV_MODEL == 'sgen' else 'gen'
    floor, ceil = gen_module.V_SET_FLOOR, gen_module.V_SET_CEIL
    nb_flag = getattr(gen_module, 'NUMBA_PF', False)

    def prep(sim):
        sim.load['p_mw'] = base_p * load_s
        sim.load['q_mvar'] = base_q * load_s
        sim.gen['vm_pu'] = np.clip(base_vg + dv, floor, ceil)
        sim.ext_grid['vm_pu'] = np.clip(base_vs + dv, floor, ceil)
        tbl = getattr(sim, pv_tbl)
        for i in pv_idx:
            tbl.at[i, 'p_mw'] = pv_mw / len(pv_idx)
        return sim

    def trip(sim, k):
        typ, bi = branches[k]
        if typ == 'line':
            sim.line.at[bi, 'in_service'] = False
        else:
            sim.trafo.at[bi, 'in_service'] = False
        return sim

    # ---- oracle: one solve per contingency (timed, and used as ground truth)
    t_oracle, truth = [], None
    for _ in range(n_repeat):
        lab = np.zeros(n_branch, np.int64)
        t0 = time.time()
        for k in range(n_branch):
            sim = trip(prep(deepcopy(net0)), k)
            try:
                pp.runpp(sim, enforce_q_lims=True, numba=nb_flag, max_iteration=30)
                conv = not sim.res_bus['vm_pu'].isna().any()
                vmin = float(sim.res_bus['vm_pu'].min()) if conv else 0.0
                vmax = float(sim.res_bus['vm_pu'].max()) if conv else 0.0
            except Exception:
                conv, vmin, vmax = False, 0.0, 0.0
            lab[k] = gen_module.risk_of(vmin, vmax, conv)
        t_oracle.append(time.time() - t0)
        if truth is None:
            truth = lab
    t_or = float(np.median(t_oracle))

    # ---- surrogate: base solve + features + normalisation + batched forward
    dev = d['dev']
    t_e2e, t_fwd, pred = [], [], None
    for _ in range(n_repeat):
        t0 = time.time()
        sim = prep(deepcopy(net0))
        pp.runpp(sim, enforce_q_lims=True, numba=nb_flag, max_iteration=30)
        nf_np = np.zeros((len(bus_ids), 4), np.float32)
        for ii, bb in enumerate(bus_ids):
            nf_np[ii, 0] = sim.res_bus.at[bb, 'vm_pu']
            nf_np[ii, 1] = sim.res_bus.at[bb, 'va_degree'] / 180.0
            nf_np[ii, 2] = sim.res_bus.at[bb, 'p_mw'] / 100.0
            nf_np[ii, 3] = sim.res_bus.at[bb, 'q_mvar'] / 100.0
        ef_np = np.zeros((n_branch, 2), np.float32)
        for k, (typ, bi) in enumerate(branches):
            if typ == 'line' and bi in sim.res_line.index:
                ef_np[k, 0] = (sim.res_line.at[bi, 'loading_percent'] or 0) / 100.0
                ef_np[k, 1] = (sim.res_line.at[bi, 'p_from_mw'] or 0) / 100.0
            elif typ == 'trafo' and bi in sim.res_trafo.index:
                ef_np[k, 0] = (sim.res_trafo.at[bi, 'loading_percent'] or 0) / 100.0
                ef_np[k, 1] = (sim.res_trafo.at[bi, 'p_hv_mw'] or 0) / 100.0
        # SAME normalisation as training. This is preprocessing, so it is inside
        # the end-to-end clock.
        NF1 = (torch.tensor(np.nan_to_num(nf_np), device=dev) - d['nmean']) / d['nstd']
        EF1 = (torch.tensor(np.nan_to_num(ef_np), device=dev) - d['emean']) / d['estd']
        NF = NF1.unsqueeze(0).expand(n_branch, -1, -1).contiguous()
        EF = EF1.unsqueeze(0).expand(n_branch, -1, -1).contiguous()
        INS = torch.ones(n_branch, n_branch, device=dev)
        INS[torch.arange(n_branch), torch.arange(n_branch)] = 0.0
        t_prep = time.time()
        cf, adj = cells_and_adj(EF, INS, d['ctx'])
        net.eval()
        r = net(NF, cf, adj)[0]
        if str(dev).startswith('cuda'):
            torch.cuda.synchronize()
        t1 = time.time()
        t_e2e.append(t1 - t0)
        t_fwd.append(t1 - t_prep)
        if pred is None:
            pred = r.float().argmax(1).cpu().numpy()

    n_unstable = int((truth == 2).sum())
    false_safe = int(((truth == 2) & (pred == 0)).sum())
    out = dict(
        n_contingencies=n_branch,
        oracle_s=t_or,
        surrogate_end_to_end_s=float(np.median(t_e2e)),
        surrogate_forward_only_s=float(np.median(t_fwd)),
        sweep_accuracy=float((pred == truth).mean()),
        sweep_n_unstable_true=n_unstable,
        sweep_n_flagged_not_stable=int((pred != 0).sum()),
        sweep_false_safe=false_safe,
        sweep_operating_point=dict(load_scale=load_s, schedule_shift=dv, pv_mw=pv_mw),
    )
    out['speedup_end_to_end'] = out['oracle_s'] / out['surrogate_end_to_end_s']
    out['speedup_forward_only'] = out['oracle_s'] / out['surrogate_forward_only_s']
    if verbose:
        print(f"  N-1 sweep of {n_branch} contingencies (load {load_s}x, PV {pv_mw:.0f} MW)")
        print(f"    oracle (pandapower)     : {out['oracle_s']*1000:8.1f} ms")
        print(f"    surrogate, end to end   : {out['surrogate_end_to_end_s']*1000:8.1f} ms"
              f"   -> {out['speedup_end_to_end']:6.1f}x")
        print(f"    surrogate, forward only : {out['surrogate_forward_only_s']*1000:8.1f} ms"
              f"   -> {out['speedup_forward_only']:6.1f}x")
        print(f"    sweep accuracy          : {out['sweep_accuracy']*100:.1f}%  "
              f"({n_unstable} truly unstable, {false_safe} cleared as Stable)")
    return out


# ============================================================================
# Unstable-class decomposition (reviewer R1 #4)
# ============================================================================
def unstable_breakdown(npz_path, gen_module, verbose=True):
    """
    Split the Unstable class into (a) converged with V_min < 0.90, (b) divergence
    where the trip set ISLANDS part of the network, and (c) divergence with the
    network still fully connected.  Distinguishing (b) from (c) is exactly what
    R1 #4 asked for: physical instability versus numerical solver failure.

    The islanding test applies each scenario's ACTUAL trip set.  With k up to 4,
    branches that island nothing on their own can island jointly, so testing only
    the single-branch islanders would misfile those as "still connected".  Trip
    sets are cached, so the divergent scenarios cost far fewer topology checks
    than there are scenarios.
    """
    import pandapower.topology as top
    D = np.load(npz_path)
    risk, div, ins = D['t_risk'], D['t_div'], D['in_service']
    net0, _ = gen_module.build_net()
    branches = ([('line', i) for i in net0.line.index]
                + [('trafo', i) for i in net0.trafo.index])

    def islands(trip_idx):
        sim = deepcopy(net0)
        for k in trip_idx:
            typ, bi = branches[k]
            if typ == 'line':
                sim.line.at[bi, 'in_service'] = False
            else:
                sim.trafo.at[bi, 'in_service'] = False
        try:
            return len(top.unsupplied_buses(sim)) > 0
        except Exception:
            return False

    unst = risk == 2
    diverged = unst & (div == 1)
    cache, n_isl = {}, 0
    for i in np.where(diverged)[0]:
        key = tuple(np.where(ins[i] == 0)[0].tolist())
        if key not in cache:
            cache[key] = islands(key)
        n_isl += bool(cache[key])

    solo = sum(1 for k in range(len(branches)) if islands((k,)))
    # Split the converged violations by SIDE.  Eq. (risk) is two-sided, so the
    # overvoltage branch has to be reported separately -- it is the direct answer
    # to R1 #5, which asked whether the overvoltage case is ever exercised.
    vbus = D['t_vbus']
    v_hi = float(D['v_hi']) if 'v_hi' in D else 1.05
    conv_unst = unst & (div == 0)
    vmin_s = vbus.min(axis=1)
    vmax_s = vbus.max(axis=1)
    n_under = int((conv_unst & (vmin_s < 0.90)).sum())
    n_over = int((conv_unst & (vmax_s >= v_hi)).sum())
    n_both = int((conv_unst & (vmin_s < 0.90) & (vmax_s >= v_hi)).sum())

    out = dict(
        n_unstable=int(unst.sum()),
        voltage_violation=int(conv_unst.sum()),
        undervoltage=n_under,
        overvoltage=n_over,
        both_sides=n_both,
        overvoltage_threshold=v_hi,
        divergence_islanding=int(n_isl),
        divergence_connected=int(diverged.sum()) - int(n_isl),
        n_islanding_branches_solo=int(solo),
        n_branches=len(branches),
        distinct_trip_sets_evaluated=len(cache),
    )
    if verbose:
        print(f"  Unstable = {out['n_unstable']}: "
              f"{out['voltage_violation']} voltage violation "
              f"({n_under} undervoltage, {n_over} overvoltage, {n_both} both), "
              f"{out['divergence_islanding']} islanding divergence, "
              f"{out['divergence_connected']} connected divergence")
    return out

## Measure

One 150-epoch seed per system, on the same GPU, with peak memory reset between them.

In [ ]:
"""
R3 #5, the half that needs a GPU: training time and peak GPU memory for BOTH
systems on the SAME A100.

Paste as a new cell at the end of train_118_colab.ipynb, after training has run
(it reuses build_data / train_eval / CSGNN118 already defined there, and imports
boost_core from the Drive folder for the 27-bus half).

Takes roughly 8-10 minutes: one 150-epoch seed per system.
Everything else R3 #5 asks for is already measured on CPU in compute_burden.py.
"""
import json
import os
import time

import torch

results = {}


def train_and_measure(name, fn):
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    res = fn()
    torch.cuda.synchronize()
    dt = time.time() - t0
    peak = torch.cuda.max_memory_allocated() / 1e9
    # boost_core (27-bus) reports false_safe as a RATE and has no count;
    # train118 (118-bus) reports both. Handle either shape.
    fs_rate = res.get('false_safe', float('nan'))
    fs_n = res.get('false_safe_n')
    results[name] = dict(
        train_seconds=dt,
        peak_gpu_gb=peak,
        test_accuracy=res.get('acc'),
        false_safe_rate=fs_rate,
        false_safe_n=fs_n,
        params=res.get('params'),
    )
    shown = f"{fs_n}" if fs_n is not None else f"{fs_rate*100:.3f}%"
    print(f"  {name:10s} {dt:7.1f} s   peak GPU {peak:5.2f} GB   "
          f"acc {res.get('acc', float('nan'))*100:.2f}%   false-safe {shown}")
    return res


print("training both systems on the same GPU for a like-for-like comparison")
print(f"device: {torch.cuda.get_device_name(0)}")
print()

# ---------------------------------------------------------------- 27-bus
import boost_core                                    # already on sys.path via FOLDER

d27 = boost_core.build_data(os.path.join(FOLDER, 'contingency_data.npz'),
                            os.path.join(FOLDER, 'grouped_split.npz'), 'cuda')
train_and_measure('27-bus', lambda: boost_core.train_eval(
    d27, seed=0, alpha=1.0, branched=False, stack=False, vbus=False,
    epochs=150, bs=256, verbose=False)[0])
del d27
torch.cuda.empty_cache()

# ---------------------------------------------------------------- 118-bus
d118 = build_data(NPZ, SPLIT, dev='cuda')
# vmax=True: measure the DEPLOYED configuration. Without it this trains the
# 163,592-parameter model, not the 188,297-parameter one the paper reports.
train_and_measure('118-bus', lambda: train_eval(
    d118, seed=0, epochs=150, bs=256, vmax=True, verbose=False)[0])

# ---------------------------------------------------------------- report
print()
print("R3 #5 -- separate costs, both systems")
print(f"  {'stage':24s} {'27-bus':>14s} {'118-bus':>14s}")
print(f"  {'dataset generation':24s} {'~26 min CPU':>14s} {'~82 min CPU':>14s}")
print(f"  {'training (150 ep, 1 seed)':24s} "
      f"{results['27-bus']['train_seconds']:11.0f} s {results['118-bus']['train_seconds']:11.0f} s")
print(f"  {'peak GPU memory':24s} "
      f"{results['27-bus']['peak_gpu_gb']:11.2f} GB {results['118-bus']['peak_gpu_gb']:11.2f} GB")
print(f"  {'inference / contingency':24s} {'1.14 ms':>14s} {'2.31 ms':>14s}")
print(f"  {'parameters':24s} {results['27-bus']['params']:14,d} "
      f"{results['118-bus']['params']:14,d}")
print("  (inference measured single-threaded on CPU, same machine, see compute_burden.json)")

results['device'] = torch.cuda.get_device_name(0)
path = os.path.join(OUT_DIR, 'compute_burden_gpu.json')
with open(path, 'w') as f:
    json.dump(results, f, indent=2, default=float)
print(f"\nsaved {path}")

---
**Done.** `compute_burden_gpu.json` is written to `smart_load_shield_boost/scale118/` on your
Drive.

Report training time and peak GPU memory per system, and quote the device name printed above.